# Replicated lead variants

The non-redundant set behind Figure 2 and Extended Data Figures 4 and 9: replicated
credible sets with lead-variant PIP >= 0.5 across four study types — disease GWAS,
measurement GWAS, cis-pQTL and eQTL. trans-QTLs are excluded, and rescaled effect sizes
above 3 and missing or zero MAF are dropped.

Writes `replicated_lead_variants` and `replicated_lead_variants_common` (MAF >= 0.01).

In [ ]:
from gentropy.common.session import Session
from pyspark.sql import functions as f

from manuscript_methods import group_statistics, paper
from manuscript_methods.datasets import LeadVariantEffect
from manuscript_methods.locus_statistics import LocusStatistics
from manuscript_methods.study_statistics import StudyStatistics, StudyType

session = Session(extended_spark_conf={"spark.driver.memory": "40G"})

PIP_THRESHOLD = 0.5
RESCALED_BETA_ABS_THRESHOLD = 3
MAF_COMMON = 0.01

In [ ]:
full = LeadVariantEffect.from_parquet(session=session, path=paper.derived("lead_variant_effect"))
qualifying_diseases = LeadVariantEffect.from_parquet(session=session, path=paper.derived("qualifying_credible_sets"))
qualifying_measurements = LeadVariantEffect.from_parquet(
    session=session, path=paper.derived("qualifying_measurement_credible_sets")
)
replicated = session.spark.read.parquet(paper.derived("replicated_gwas_cs")).unionByName(
    session.spark.read.parquet(paper.derived("replicated_molqtl_cs"))
)

## Union of the four study types

In [ ]:
study_stats = StudyStatistics()

molqtl = LeadVariantEffect(
    full.df.filter(study_stats.study_type.isin(StudyType.CIS_PQTL, StudyType.EQTL)).filter(~f.col("isTransQtl"))
)
diseases = LeadVariantEffect(
    qualifying_diseases.df.withColumn(study_stats.name, study_stats.transform_study_type(StudyType.GWAS_DISEASE).col)
)
measurements = LeadVariantEffect(
    qualifying_measurements.df.withColumn(
        study_stats.name, study_stats.transform_study_type(StudyType.GWAS_MEASUREMENT).col
    )
)
qualified = LeadVariantEffect(molqtl.df.unionByName(measurements.df).unionByName(diseases.df))
print("molQTL:", molqtl.df.count(), "measurement:", measurements.df.count(), "disease:", diseases.df.count())

## Keep replicated sets, then filter MAF, effect size and PIP

In [ ]:
locus_stats = LocusStatistics()

filtered = (
    qualified.filter_by_study_locus_id(replicated.distinct())
    .maf_filter(remove_null=True, remove_zero=True, threshold=None)
    .effect_size_filter(effect_size_threshold=RESCALED_BETA_ABS_THRESHOLD)
)
final = filtered.filter(locus_stats.col.getField("leadVariantPIP") >= PIP_THRESHOLD)
common = final.maf_filter(threshold=MAF_COMMON)

final.df.write.mode("overwrite").parquet(paper.derived("replicated_lead_variants"))
common.df.write.mode("overwrite").parquet(paper.derived("replicated_lead_variants_common"))

print("replicated lead variants:", session.spark.read.parquet(paper.derived("replicated_lead_variants")).count())
print("of those common:", session.spark.read.parquet(paper.derived("replicated_lead_variants_common")).count())

In [ ]:
group_statistics(
    session.spark.read.parquet(paper.derived("replicated_lead_variants")).select("studyStatistics.studyType"),
    [f.col("studyType")],
).show()
print("baseline:", session.spark.read.parquet(paper.baseline("qualified_lead_variant_effect")).count())